# task 1 -> direct preference optimization

## imports

In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#the scripts use paths relative to the repo root (configs/, data/, results/), so the notebook runs from there
if not Path('configs').exists():
    os.chdir('..')

from common.data import load_yaml, read_jsonl, repo_path

## global env variables ->

In [ ]:
#papermill overrides these two for the smoke test, the defaults are the real run
CONFIG = 'configs/dpo.yaml'
RUN_SCRIPTS = True #False -> only rebuild the tables and figures from results already on disk

In [ ]:
cfg = load_yaml(CONFIG)
RESULTS = repo_path(cfg['results_dir'])
FIGURES = RESULTS / 'figures'
STANDARD = cfg['standard_output']

BETAS = [float(beta) for beta in cfg['betas']]
BETA_RUNS = [f'beta_{beta:g}' for beta in BETAS]
ALL_RUNS = ['base', 'standard'] + BETA_RUNS + ['length_balanced'] #base = the untrained reference policy
TRAINED_RUNS = ALL_RUNS[1:]

#one fixed colour per model, the same in every figure. base is the reference so it stays neutral grey
COLORS = {'base': '#52514e', 'standard': '#2a78d6', 'length_balanced': '#eb6834',
          BETA_RUNS[0]: '#1baf7a', BETA_RUNS[1]: '#eda100', BETA_RUNS[2]: '#e87ba4'}
STRATA = ['preferred_longer', 'length_matched', 'rejected_longer']

plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
                     'grid.color': '#e6e5e1', 'grid.linewidth': 0.6, 'axes.axisbelow': True, 'lines.linewidth': 2})


def run(command):
    #runs one of the release's entry points as its own process, exactly as it would be typed in a terminal.
    #each script loads its own models and writes its results to disk, nothing is passed through notebook state
    if RUN_SCRIPTS:
        get_ipython().system(command)
        assert get_ipython().user_ns['_exit_code'] == 0, f'failed -> {command}'


def load_result(name):
    return json.loads((RESULTS / name).read_text())

## standard dpo ->

one epoch on the fixed preference set from the common initialization, then the held-out evaluation. the untrained base policy goes through the same evaluation as the reference row

In [ ]:
run(f'python -m task1_dpo.train --config {CONFIG} --run-name standard --output {STANDARD}')

In [ ]:
run(f'python -m task1_dpo.evaluate --config {CONFIG} --name base')
run(f'python -m task1_dpo.evaluate --config {CONFIG} --adapter {STANDARD} --name standard')

## regularization strength study ->

three short forks from the original initialization, only beta changes

In [ ]:
run(f'python -m task1_dpo.ablate_beta --config {CONFIG}')

## length confounding study ->

one more model on the length-balanced pairs, then both models on the length-stratified held-out set

In [ ]:
run(f'python -m task1_dpo.analyze_length --config {CONFIG}')

## dpo summary results

In [ ]:
#one row per condition. the budget columns are there because the standard run and the beta forks are NOT matched ->
#standard and length_balanced see every usable pair once, each beta fork only the first short_ablation_examples
FIGURES.mkdir(parents=True, exist_ok=True)

rows = []
for name in ALL_RUNS:
    ev = load_result(f'eval_{name}.json')
    train = load_result(f'train_summary_{name}.json') if name != 'base' else {}
    rows.append({
        'model': name,
        'beta': ev['beta'] if name != 'base' else np.nan,
        'train_pairs': train.get('n_pairs', 0),
        'optimizer_steps': train.get('optimizer_steps', 0),
        'heldout_dpo_loss': ev['heldout_dpo_loss'],
        #the base policy IS the reference, so its margin is exactly 0 and its accuracy is 0 by definition, not by failure
        'preference_accuracy': ev['heldout_preference_accuracy'] if name != 'base' else np.nan,
        'kl_token': ev['kl_token_mean'],
        'kl_sequence': ev['kl_sequence_mean'],
        'reward_mean': ev['reward_mean'],
        'length_mean': ev['length_mean'],
        'length_std': ev['length_std'],
        'length_iqr': ev['length_iqr'],
        'truncation_rate': ev['truncation_rate'],
        'word_limit_compliance': ev['word_limit']['compliance_rate'],
        'train_minutes': train.get('wall_clock_s', 0) / 60,
        'peak_vram_gb': train.get('peak_vram_gb'),
    })

summary = pd.DataFrame(rows)
summary.to_csv(RESULTS / 'dpo_summary.csv', index=False)
display(summary.round(4))

In [ ]:
#training curves, one line per trained run. x is in pairs seen rather than steps, so the short forks and the
#full runs sit on a common axis
logs = {name: pd.DataFrame(read_jsonl(RESULTS / f'train_log_{name}.jsonl')) for name in TRAINED_RUNS}
effective_batch = int(cfg['batch_size']) * int(cfg['grad_accum_steps'])

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for axis, column, title in zip(axes, ['loss', 'acc', 'grad_norm'],
                               ['training dpo loss', 'training preference accuracy', 'gradient norm before clipping']):
    for name, log in logs.items():
        #rolling mean over 5 optimizer steps, a single step of 16 pairs is too noisy to read
        axis.plot(log['step'] * effective_batch, log[column].rolling(5, min_periods=1).mean(), color=COLORS[name], label=name)
    axis.set_title(title)
    axis.set_xlabel('preference pairs seen')
axes[0].axhline(np.log(2), color='#52514e', linestyle=':', linewidth=1) #log 2 = the loss at initialization, margin 0
axes[0].legend(frameon=False)
fig.tight_layout()
fig.savefig(FIGURES / 'training_curves.png', dpi=200)
plt.show()

In [ ]:
#the beta trade-off. one small panel per metric (they live on different scales, so no shared or double axis),
#beta on a log axis since the three values are spaced by ~3x
beta_frame = summary[summary['model'].isin(BETA_RUNS)].sort_values('beta')
panels = [('preference_accuracy', 'held-out preference accuracy'), ('heldout_dpo_loss', 'held-out dpo loss'),
          ('kl_token', 'kl from reference (per token)'), ('reward_mean', 'reward model score'),
          ('length_mean', 'generated length (tokens)')]

fig, axes = plt.subplots(1, len(panels), figsize=(21, 4))
base_row = summary[summary['model'] == 'base'].iloc[0]
for axis, (column, title) in zip(axes, panels):
    axis.plot(beta_frame['beta'], beta_frame[column], color=COLORS['standard'], marker='o', markersize=8)
    for beta, value in zip(beta_frame['beta'], beta_frame[column]):
        axis.annotate(f'{value:.3g}', (beta, value), textcoords='offset points', xytext=(0, 9), ha='center', color='#0b0b0b')
    if column in ('reward_mean', 'length_mean'): #the untrained policy as the reference level, where it is meaningful
        axis.axhline(base_row[column], color=COLORS['base'], linestyle='--', linewidth=1.2, label='base')
        axis.legend(frameon=False)
    axis.ticklabel_format(axis='y', useOffset=False) #plain tick values, no '+6.93e-1' offset notation
    axis.set_xscale('log')
    axis.set_xticks(BETAS, [f'{beta:g}' for beta in BETAS])
    axis.minorticks_off()
    axis.set_xlabel('beta')
    axis.set_title(title)
    axis.margins(y=0.25)
fig.tight_layout()
fig.savefig(FIGURES / 'beta_tradeoff.png', dpi=200)
plt.show()

In [ ]:
#generated length distribution per model on the held-out prompts, same prompts and decoding for every box
generations = {name: pd.DataFrame(read_jsonl(RESULTS / f'eval_{name}_generations.jsonl')) for name in ALL_RUNS}

fig, axis = plt.subplots(figsize=(9, 4.5))
boxes = axis.boxplot([generations[name]['response_tokens'] for name in ALL_RUNS], orientation='horizontal', patch_artist=True,
                     widths=0.55, medianprops={'color': '#0b0b0b'}, flierprops={'markersize': 3})
for box, name in zip(boxes['boxes'], ALL_RUNS):
    box.set_facecolor(COLORS[name])
    box.set_edgecolor('#fcfcfb')
axis.set_yticks(range(1, len(ALL_RUNS) + 1), ALL_RUNS)
axis.invert_yaxis()
axis.axvline(int(cfg['max_generation_tokens']), color='#52514e', linestyle=':', linewidth=1) #the generation cap
axis.set_xlabel('generated response tokens')
axis.set_title('response length on the held-out prompts')
axis.grid(axis='y', visible=False)
fig.tight_layout()
fig.savefig(FIGURES / 'length_distribution.png', dpi=200)
plt.show()

## length confounding diagnostics

In [ ]:
#dataset side -> how tied length and preference are in the two training files, before any model is involved
display(pd.DataFrame(load_result('length_dataset_diagnostics.json')).T.round(3))

#model side -> preference accuracy per stratum for the standard and the length-balanced model
strata = pd.read_csv(RESULTS / 'length_strata_accuracy.csv')
display(strata.round(4))

accuracy = strata[strata['stratum'].isin(STRATA)].pivot(index='stratum', columns='model', values='preference_accuracy').loc[STRATA]

fig, axis = plt.subplots(figsize=(8, 4.5))
positions = np.arange(len(STRATA))
for offset, name in zip([-0.2, 0.2], ['standard', 'length_balanced']):
    bars = axis.bar(positions + offset, accuracy[name], width=0.36, color=COLORS[name], label=name)
    axis.bar_label(bars, fmt='%.2f', padding=3, color='#0b0b0b')
axis.axhline(0.5, color='#52514e', linestyle=':', linewidth=1) #a coin flip
axis.set_xticks(positions, [stratum.replace('_', ' ') for stratum in STRATA])
axis.set_ylim(0, 1.08)
axis.set_ylabel('held-out preference accuracy')
axis.set_title('preference accuracy by length stratum')
axis.grid(axis='x', visible=False)
axis.legend(frameon=False, loc='upper left', bbox_to_anchor=(1.0, 1.0)) #outside the plot, clear of the bars
fig.tight_layout()
fig.savefig(FIGURES / 'length_strata_accuracy.png', dpi=200)
plt.show()

In [ ]:
#generated length and explicit word-limit compliance on the common prompt set, base included as the reference
word_limit = {name: pd.DataFrame(read_jsonl(RESULTS / f'eval_{name}_word_limit.jsonl')) for name in ALL_RUNS}

compare = pd.DataFrame([{
    'model': name,
    'heldout_length_mean': generations[name]['response_tokens'].mean(),
    'heldout_length_std': generations[name]['response_tokens'].std(ddof=0),
    'word_limit_compliance': word_limit[name]['compliant'].mean(),
    'word_limit_words_mean': word_limit[name]['words'].mean(),
    'word_limit_tokens_mean': word_limit[name]['response_tokens'].mean(),
} for name in ['base', 'standard', 'length_balanced']])
compare.to_csv(RESULTS / 'length_generation_compare_with_base.csv', index=False)
display(compare.round(3))

#compliance per prompt, the limit differs from prompt to prompt (25 to 45 words)
per_prompt = pd.DataFrame({name: word_limit[name].groupby('prompt_id')['compliant'].mean()
                           for name in ['base', 'standard', 'length_balanced']})
display(per_prompt.round(2))

## qualitative candidates ->

shortlists to read through and pick the report examples from, nothing here is selected automatically

In [ ]:
pd.set_option('display.max_colwidth', 400)

#(i) preference/reward versus quality -> prompts where the dpo response scores clearly higher on the reward model
#than the base response to the same prompt, sorted by the gain. the largest gains that come with a much longer or
#truncated response are the first places to look for a higher score without a better answer
paired = generations['base'].merge(generations['standard'], on='prompt_id', suffixes=('_base', '_dpo'))
paired['reward_gain'] = paired['reward_dpo'] - paired['reward_base']
paired['length_change'] = paired['response_tokens_dpo'] - paired['response_tokens_base']
columns = ['prompt_id', 'reward_base', 'reward_dpo', 'reward_gain', 'response_tokens_base', 'response_tokens_dpo',
           'truncated_dpo', 'prompt_base', 'response_base', 'response_dpo']
display(paired.sort_values('reward_gain', ascending=False)[columns].head(8))
print('correlation of reward gain with length change', round(paired['reward_gain'].corr(paired['length_change']), 3))

In [ ]:
#held-out pairs the standard model is most confident about (largest margin) even though the human scores of the
#two responses are tied or nearly tied -> a strong preference signal the annotations themselves do not support
pairs = pd.DataFrame(read_jsonl(RESULTS / 'eval_standard_pairs.jsonl'))
pairs['score_gap'] = pairs['score_chosen'] - pairs['score_rejected']
pairs['token_gap'] = pairs['chosen_tokens'] - pairs['rejected_tokens']
display(pairs[pairs['score_gap'].abs() <= 0.5].sort_values('margin', key=abs, ascending=False)
        [['prompt_id', 'margin', 'correct', 'score_chosen', 'score_rejected', 'chosen_tokens', 'rejected_tokens']].head(8))
print('correlation of the learned margin with the chosen-minus-rejected token gap', round(pairs['margin'].corr(pairs['token_gap']), 3))

In [ ]:
#(ii) length bias / instruction compliance -> word-limit prompts where the base policy complies and the standard
#dpo model does not, with the length-balanced model's answer to the same prompt next to it
first = {name: word_limit[name][word_limit[name]['sample'] == 0].set_index('prompt_id') for name in ['base', 'standard', 'length_balanced']}
violations = pd.DataFrame({
    'prompt': first['base']['prompt'],
    'base_words': first['base']['words'], 'standard_words': first['standard']['words'],
    'balanced_words': first['length_balanced']['words'],
    'base_ok': first['base']['compliant'], 'standard_ok': first['standard']['compliant'],
    'balanced_ok': first['length_balanced']['compliant'],
    'standard_response': first['standard']['response'], 'balanced_response': first['length_balanced']['response'],
})
display(violations.sort_values('standard_words', ascending=False))